# Data Collection

Pull KIT (CHEMBL1936, UniProt P10721) bioactivity records from ChEMBL and save the raw, unmodified pull to `data/raw/`.

See [Design Doc.md](../Design%20Doc.md) §4.1 and §5.1, and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 1.

In [1]:
import pandas as pd
from pathlib import Path
from chembl_webresource_client.new_client import new_client
from chembl_webresource_client.settings import Settings

# Default page size is 20 records/request, which makes a ~8.7k-record pull take
# ~10+ minutes (hundreds of round trips). Raising it cuts that to under 2 minutes.
Settings.Instance().MAX_LIMIT = 1000

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1. Resolve the KIT target

Confirmed in Phase 1 step 1: human KIT resolves to `CHEMBL1936` ("Mast/stem cell growth factor receptor Kit", *Homo sapiens*), which cross-references UniProt `P10721` (Design Doc §4.1).

In [2]:
KIT_TARGET_CHEMBL_ID = "CHEMBL1936"

target = new_client.target
t = target.get(KIT_TARGET_CHEMBL_ID)

uniprot_ids = [
    xref["xref_id"]
    for comp in t["target_components"]
    for xref in comp.get("target_component_xrefs", [])
    if xref["xref_src_db"] == "UniProt"
]

print(t["target_chembl_id"], "-", t["pref_name"], "-", t["organism"])
print("UniProt cross-refs:", uniprot_ids)
assert "P10721" in uniprot_ids, "Expected UniProt P10721 among KIT target cross-references"

CHEMBL1936 - Mast/stem cell growth factor receptor Kit - Homo sapiens
UniProt cross-refs: ['B5A956', 'D5LXN2', 'D5M931', 'F5H8F8', 'Q6IQ28', 'Q99662', 'Q9UM99', 'P10721']


## 2. Pull bioactivity records

Restrict `standard_type` to the four potency measures named in Design Doc §4.1/§5.1 (IC50, Ki, Kd, EC50) — ChEMBL also carries single-concentration `%Inhibition`-type records for this target which are not directly comparable potency values and are out of scope for the regression target built in Phase 2.

Fields pulled: compound identity + structure, standard potency value/units/relation, `pchembl_value` (ChEMBL's own -log10 molar normalization, useful as a cross-check in Phase 2), assay metadata, document metadata, and `target_organism` (for the wild-type vs. mutant segmentation in Phase 2/5 — see note below).

In [3]:
STANDARD_TYPES = ["IC50", "Ki", "Kd", "EC50"]

FIELDS = [
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "document_chembl_id",
    "document_year",
    "target_chembl_id",
    "target_organism",
]

activity = new_client.activity

records = []
for stype in STANDARD_TYPES:
    res = activity.filter(
        target_chembl_id=KIT_TARGET_CHEMBL_ID, standard_type=stype
    ).only(FIELDS)
    records.extend(list(res))

df_raw = pd.DataFrame.from_records(records)
print("Total records pulled:", len(df_raw))
df_raw["standard_type"].value_counts()

Total records pulled: 8703


standard_type
IC50    4485
Ki      2775
Kd      1237
EC50     206
Name: count, dtype: int64

## 3. Sanity checks

Spot-check that well-known KIT inhibitors are present, and inspect basic shape/nulls before saving.

In [4]:
# Known KIT inhibitors, resolved by name to their ChEMBL parent-molecule IDs.
# Note some compounds have multiple ChEMBL entries (parent vs. salt form, e.g.
# CHEMBL1421 "DASATINIB ANHYDROUS" vs. CHEMBL5416410 "DASATINIB") — check all forms.
molecule = new_client.molecule
spot_check_names = ["imatinib", "dasatinib", "avapritinib"]

spot_check_ids = {}
for name in spot_check_names:
    hits = molecule.filter(pref_name__icontains=name).only(
        ["molecule_chembl_id", "pref_name"]
    )
    spot_check_ids[name] = [h["molecule_chembl_id"] for h in hits]
    print(name, "->", spot_check_ids[name])

for name, ids in spot_check_ids.items():
    present = df_raw["molecule_chembl_id"].isin(ids).sum()
    print(f"{name}: {present} KIT bioactivity records found")
    assert present > 0, f"Expected at least one KIT record for {name}"

imatinib -> ['CHEMBL941', 'CHEMBL1642']


dasatinib -> ['CHEMBL1421', 'CHEMBL5314601', 'CHEMBL5416410']


avapritinib -> ['CHEMBL4204794']
imatinib: 69 KIT bioactivity records found
dasatinib: 36 KIT bioactivity records found
avapritinib: 7 KIT bioactivity records found


In [5]:
print("Shape:", df_raw.shape)
print("\nNulls per column:\n", df_raw.isna().sum())
df_raw.head()

Shape: (8703, 18)

Nulls per column:
 assay_chembl_id          0
assay_description        0
assay_type               0
canonical_smiles         4
document_chembl_id       0
document_year          667
molecule_chembl_id       0
pchembl_value         2992
relation               156
standard_relation      156
standard_type            0
standard_units         154
standard_value         156
target_chembl_id         0
target_organism          0
type                     0
units                  868
value                  156
dtype: int64


,assay_chembl_id,assay_description,assay_type,canonical_smiles,document_chembl_id,document_year,molecule_chembl_id,pchembl_value,relation,standard_relation,standard_type,standard_units,standard_value,target_chembl_id,target_organism,type,units,value
0,CHEMBL820421,Inhibition of c-Kit autophosphorylation in int...,B,COc1cc2c(Oc3ccc(Nc4ccc(C(C)(C)C)cc4)cc3)ccnc2c...,CHEMBL1146677,2004.0,CHEMBL352308,7.00,=,=,IC50,nM,100.0,CHEMBL1936,Homo sapiens,IC50,nM,100.0
1,CHEMBL702237,Inhibition of KIT kinase activity,B,O=C(Cc1ccc2ccccc2c1)Nc1cc(C2CC2)n[nH]1,CHEMBL1148336,2004.0,CHEMBL115220,None,>,>,IC50,nM,10000.0,CHEMBL1936,Homo sapiens,IC50,nM,10000.0
2,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(OC(C)C)cc4)CC3)ncnc...,CHEMBL1135998,2002.0,CHEMBL330863,7.68,=,=,IC50,nM,21.0,CHEMBL1936,Homo sapiens,IC50,uM,0.021
3,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(OC(C)C)cc4)CC3)ncnc...,CHEMBL1135998,2002.0,CHEMBL124660,6.77,=,=,IC50,nM,170.0,CHEMBL1936,Homo sapiens,IC50,uM,0.17
4,CHEMBL766073,Inhibition of chimeric PDGF receptor with c-ki...,B,COc1cc2c(N3CCN(C(=O)Nc4ccc(C#N)cc4)CC3)ncnc2cc...,CHEMBL1135998,2002.0,CHEMBL126699,8.22,=,=,IC50,nM,6.0,CHEMBL1936,Homo sapiens,IC50,uM,0.006


In [6]:
# ChEMBL's target index for CHEMBL1936 is wild-type human KIT; there is no separate
# target entry for the D816V mutant, and assay_description doesn't support server-side
# substring filtering, so search it locally for mutant-KIT assays.
mutant_mask = df_raw["assay_description"].str.contains("D816V", case=False, na=False)
print("Records with 'D816V' in assay_description:", mutant_mask.sum())
df_raw.loc[mutant_mask, ["assay_chembl_id", "assay_description"]].drop_duplicates()

Records with 'D816V' in assay_description: 2291


,assay_chembl_id,assay_description
86,CHEMBL873572,Inhibitory concentration against c-Kit D816V t...
88,CHEMBL883229,Inhibitory concentration against IL-3 independ...
120,CHEMBL883241,Inhibitory concentration against IL-3 independ...
672,CHEMBL1244577,Inhibition of cKIT D816V mutant
732,CHEMBL1787710,Inhibition of human KIT D816V using gamma-33P-...
759,CHEMBL1932833,Inhibition of c-KIT (D816V)
826,CHEMBL2319222,Inhibition of human c-Kit D816V mutant using p...
888,CHEMBL2349757,Inhibition of c-Kit D816V mutant (unknown orig...
1068,CHEMBL3368317,Inhibition of c-KIT D816V mutant (unknown origin)
1260,CHEMBL3807387,Inhibition of N-terminal GST-His-tagged c-KIT ...


## 4. Save raw, unmodified pull

No cleaning/filtering yet (that's Phase 2, `02_data_cleaning.ipynb`) — this is the raw ChEMBL response as retrieved.

In [7]:
out_path = RAW_DIR / "chembl_kit_bioactivity_raw.csv"
df_raw.to_csv(out_path, index=False)
print(f"Saved {len(df_raw)} records to {out_path}")

Saved 8703 records to ../data/raw/chembl_kit_bioactivity_raw.csv


## 5. Record count vs. Design Doc estimate

Design Doc §4.1 estimated "several hundred to a few thousand data points." Restricted to IC50/Ki/Kd/EC50 (excluding single-concentration `%Inhibition` screening data), the actual pull is **8,703 records** — at the upper end of that estimate.

**Note on wild-type vs. D816V-mutant records:** ChEMBL's target index (`CHEMBL1936`) represents wild-type human KIT — there is no separate ChEMBL target entry for the D816V mutant. Records against mutant KIT are distinguished only by free-text `assay_description` (e.g., mentioning "D816V"), not a structured field.

A local substring search found **2,291 of 8,703 records (~26%) mention "D816V"** in `assay_description` — meaningfully more mutant-KIT data than Design Doc §4.4 anticipated ("far more compounds tested against wild-type only"). This is good news for the Phase 5 selectivity analysis, though it doesn't change the plan to still treat that analysis as an anchor-point validation exercise rather than a standalone classifier (Design Doc §5.6) — record count alone doesn't establish how many *compounds* have **paired** wild-type-and-mutant measurements, which is what selectivity ratios actually require. That pairing is computed explicitly in Phase 2/5.